In [ ]:
import kagglehub

path = kagglehub.dataset_download("sumn2u/garbage-classification-v2")

print("Path to dataset files:", path)

In [ ]:
from pathlib import Path

dataset_path = Path(path)

print("Dataset path:", dataset_path)
print("\nMain folders/files:")
for item in dataset_path.iterdir():
    print(item)

In [ ]:
import pandas as pd
from pathlib import Path

image_extensions = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]

image_paths = [
    p for p in dataset_path.rglob("*")
    if p.suffix.lower() in image_extensions
]

data = pd.DataFrame({
    "image_path": image_paths,
    "label": [p.parent.name for p in image_paths]
})

print("Number of images:", len(data))
print("Number of classes:", data["label"].nunique())

data.head()

In [ ]:
class_distribution = data["label"].value_counts()
class_distribution

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
class_distribution.plot(kind="bar")
plt.title("Class distribution")
plt.xlabel("Waste category")
plt.ylabel("Number of images")
plt.xticks(rotation=45)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

classes = sorted(data["label"].unique())

plt.figure(figsize=(15, 8))

for idx, class_name in enumerate(classes):
    sample_path = data[data["label"] == class_name]["image_path"].iloc[0]
    image = Image.open(sample_path).convert("RGB")
    
    plt.subplot(2, 5, idx + 1)
    plt.imshow(image)
    plt.title(class_name)
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

train_data, temp_data = train_test_split(
    data,
    test_size=0.30,
    stratify=data["label"],
    random_state=42
)

val_data, test_data = train_test_split(
    temp_data,
    test_size=0.50,
    stratify=temp_data["label"],
    random_state=42
)

print("Train:", len(train_data))
print("Validation:", len(val_data))
print("Test:", len(test_data))

In [ ]:
import torch
from torchvision import transforms

image_size = 224

train_transforms = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_test_transforms = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
from torch.utils.data import Dataset
from PIL import Image

class WasteDataset(Dataset):
    def __init__(self, dataframe, label_to_idx, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.label_to_idx = label_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        image_path = self.dataframe.loc[index, "image_path"]
        label_name = self.dataframe.loc[index, "label"]

        image = Image.open(image_path).convert("RGB")
        label = self.label_to_idx[label_name]

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
classes = sorted(data["label"].unique())

label_to_idx = {class_name: idx for idx, class_name in enumerate(classes)}
idx_to_label = {idx: class_name for class_name, idx in label_to_idx.items()}

num_classes = len(classes)

print(label_to_idx)

In [ ]:
from torch.utils.data import DataLoader

batch_size = 32

train_dataset = WasteDataset(train_data, label_to_idx, transform=train_transforms)
val_dataset = WasteDataset(val_data, label_to_idx, transform=val_test_transforms)
test_dataset = WasteDataset(test_data, label_to_idx, transform=val_test_transforms)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)